In [1]:
print(123)

123


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [4]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [5]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [6]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


True

In [8]:
import os 

GROQ_MODEL = "llama-3.1-8b-instant"
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [9]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=groq_client,
)

In [12]:
vector_assistant.rag('the program has already begun, can I still sign up?')

"Based on the provided context, I would say that you can still sign up for the course, as the program has not necessarily begun yet. However, since you've just discovered the course, there may be limited time left to submit projects if you want to receive a certificate."

In [11]:
vector_assistant.rag('whats queen gambit?')

'The term "Queen\'s Gambit" does not appear to be related to the provided context about the LLM (Large Language Model) Zoomcamp. However, it\'s possible that the term "Queen\'s Gambit" is mentioned in a different context, such as chess. \n\nIn chess, the Queen\'s Gambit is an opening that starts with the following moves:\n\n1.d4 d5\n2.c4\n\nIt\'s a popular opening that offers White a range of pawn structures and attacking possibilities. The name "Queen\'s Gambit" comes from the fact that White appears to be sacrificing a pawn, but Black often ends up with a disadvantage in the endgame.\n\nIf you\'re referring to the Netflix series "The Queen\'s Gambit," it\'s a popular drama about a young chess prodigy named Beth Harmon.'